In [ ]:
import os
import shutil
import pandas as pd
import pickle
import json
import networkx as nx


# Paths
CSV_PATH = 'complex_cluster_all.csv'
FPLAN_SRC = '../../data/Tell2Design Data/General Data/floorplan_reoriented'
HUMAN_ANNOT_SRC = '../../data/Tell2Design Data/General Data/human_annotated_tags'
GRAPHS_PKL = '../../output/graphs_reoriented_dict.pkl'
OUT_BASE = './organized_plans'



In [15]:
# Load cluster info
df = pd.read_csv(CSV_PATH)
df['file_no'] = df['file_no'].astype(str)
# Load graphs dictionary from pickle
with open(GRAPHS_PKL, 'rb') as f:
    graphs_dict = pickle.load(f)



In [23]:
graphs_dict['14692']

{'rooms': [{'type': 'common_room',
   'polygon': '{"type": "Polygon", "coordinates": [[[100.0, 88.0], [100.0, 133.0], [135.0, 133.0], [135.0, 88.0], [100.0, 88.0]]]}',
   'area': 1575.0,
   'centroid': (117.5, 110.5)},
  {'type': 'common_room',
   'polygon': '{"type": "Polygon", "coordinates": [[[46.0, 137.0], [46.0, 143.0], [45.0, 144.0], [36.0, 144.0], [36.0, 178.0], [79.0, 178.0], [79.0, 137.0], [46.0, 137.0]]]}',
   'area': 1693.5,
   'centroid': (58.17852573565594, 158.19860249975395)},
  {'type': 'master_room',
   'polygon': '{"type": "Polygon", "coordinates": [[[36.0, 88.0], [36.0, 122.0], [45.0, 122.0], [46.0, 123.0], [46.0, 133.0], [96.0, 133.0], [96.0, 88.0], [36.0, 88.0]]]}',
   'area': 2590.5,
   'centroid': (67.05764652898411, 109.78041562117996)},
  {'type': 'living_room',
   'polygon': '{"type": "Polygon", "coordinates": [[[139.0, 78.0], [139.0, 136.0], [138.0, 137.0], [83.0, 137.0], [83.0, 151.0], [138.0, 151.0], [139.0, 152.0], [139.0, 178.0], [220.0, 178.0], [220.0, 1

In [24]:
def make_json_serializable(obj):
    """Convert non-serializable objects to JSON-friendly formats"""
    if isinstance(obj, tuple):
        return list(obj)
    elif hasattr(obj, '__geo_interface__'):
        # Shapely objects have this interface
        return obj.__geo_interface__
    elif hasattr(obj, 'wkt'):
        # Alternative for geometry objects with WKT representation
        return {'type': 'WKT', 'data': obj.wkt}
    elif str(type(obj)) == "<class 'shapely.geometry.polygon.Polygon'>":
        # Fallback for Shapely polygons
        return {'type': 'Polygon', 'wkt': str(obj)}
    else:
        # For other non-serializable objects
        return str(obj)

def process_graph_data(data):
    """Recursively process data structure to make it JSON serializable"""
    if isinstance(data, dict):
        result = {}
        for key, value in data.items():
            if key == 'polygon' and isinstance(value, str):
                # Parse the JSON string polygon
                try:
                    result[key] = json.loads(value)
                except:
                    result[key] = value
            else:
                result[key] = process_graph_data(value)
        return result
    elif isinstance(data, list):
        return [process_graph_data(item) for item in data]
    elif isinstance(data, (tuple, set)):
        return list(data)
    elif isinstance(data, (str, int, float, bool, type(None))):
        return data
    else:
        # Handle special objects like Shapely polygons
        return make_json_serializable(data)

In [25]:
# Process each entry
for _, row in df.iterrows():
    filename = row['filename']               # e.g., '62610.png'
    print(filename)
    tag = row['complex_tag']                # e.g., '0_5'
    file_no = str(row['file_no'])           # e.g., '62610'

    # Prepare output directories for this tag
    tag_dir = os.path.join(OUT_BASE, tag)
    dirs = {
        'floorplan_reoriented': os.path.join(tag_dir, 'floorplan_reoriented'),
        'human_annotation':    os.path.join(tag_dir, 'human_annotation'),
        'json':                os.path.join(tag_dir, 'json')
    }
    for d in dirs.values():
        os.makedirs(d, exist_ok=True)

    # Copy floorplan image
    src_img = os.path.join(FPLAN_SRC, filename)
    dst_img = os.path.join(dirs['floorplan_reoriented'], filename)
    if os.path.exists(src_img):
        shutil.copy2(src_img, dst_img)
    else:
        print(f"Missing image: {src_img}")

    # Copy human annotation text
    base_name = os.path.splitext(filename)[0]
    annot_fname = base_name + '.txt'
    src_txt = os.path.join(HUMAN_ANNOT_SRC, annot_fname)
    dst_txt = os.path.join(dirs['human_annotation'], annot_fname)
    if os.path.exists(src_txt):
        shutil.copy2(src_txt, dst_txt)
    else:
        print(f"Missing annotation: {src_txt}")

    # Retrieve graph and serialize as pretty JSON
    graph_obj = graphs_dict.get(row['file_no'])
    if graph_obj is None:
        print(f"No graph entry for file_no {row['file_no']}")
        continue

    try:
        # Process the entire graph object
        if isinstance(graph_obj, dict):
            # Direct dictionary - process it
            graph_data = process_graph_data(graph_obj)
        elif isinstance(graph_obj, nx.Graph):
            # NetworkX graph - convert to dict first
            graph_data = nx.node_link_data(graph_obj)
            graph_data = process_graph_data(graph_data)
        elif hasattr(graph_obj, 'to_dict'):
            # Has dict conversion method
            graph_data = graph_obj.to_dict()
            graph_data = process_graph_data(graph_data)
        else:
            # Fallback
            graph_data = {
                'type': type(graph_obj).__name__,
                'data': process_graph_data(graph_obj)
            }
        
        # Create prettified JSON with proper structure
        json_text = json.dumps(
            graph_data, 
            indent=2,
            sort_keys=False,  # Keep original order for better readability
            ensure_ascii=False,
            separators=(',', ': ')
        )
        
        # Add newline at end
        json_text += '\n'
        
        # Prepare output path
        json_fname = base_name + '.json'
        dst_json = os.path.join(dirs['json'], json_fname)
        
        # Ensure directory exists
        os.makedirs(dirs['json'], exist_ok=True)
        
        # Write JSON file
        with open(dst_json, 'w', encoding='utf-8') as jf:
            jf.write(json_text)
        
        print(f"Successfully wrote JSON to: {dst_json}")
        
    except Exception as e:
        print(f"Error serializing graph for file_no {row['file_no']}: {e}")
        import traceback
        traceback.print_exc()
        
        # Write error JSON
        error_data = {
            'error': True,
            'file_no': row['file_no'],
            'error_message': str(e),
            'graph_type': type(graph_obj).__name__
        }
        json_text = json.dumps(error_data, indent=2) + '\n'
        json_fname = base_name + '_error.json'
        dst_json = os.path.join(dirs['json'], json_fname)
        with open(dst_json, 'w', encoding='utf-8') as jf:
            jf.write(json_text)

print("Organization complete!")


62610.png
Successfully wrote JSON to: ./organized_plans/0_5/json/62610.json
47588.png
Successfully wrote JSON to: ./organized_plans/19_15/json/47588.json
22547.png
Successfully wrote JSON to: ./organized_plans/5_10/json/22547.json
15606.png
Successfully wrote JSON to: ./organized_plans/9_11/json/15606.json
56702.png
Successfully wrote JSON to: ./organized_plans/5_20/json/56702.json
36698.png
Successfully wrote JSON to: ./organized_plans/15_13/json/36698.json
22487.png
Successfully wrote JSON to: ./organized_plans/3_3/json/22487.json
28400.png
Successfully wrote JSON to: ./organized_plans/18_18/json/28400.json
64149.png
Successfully wrote JSON to: ./organized_plans/0_15/json/64149.json
33069.png
Successfully wrote JSON to: ./organized_plans/7_12/json/33069.json
47407.png
Successfully wrote JSON to: ./organized_plans/13_5/json/47407.json
61434.png
Successfully wrote JSON to: ./organized_plans/18_7/json/61434.json
28732.png
Successfully wrote JSON to: ./organized_plans/14_8/json/28732.jso